In [1]:
# 05_run_ctabgan.ipynb
# Multi-seed run of the CTAB-GAN+ baseline (vendored repo Team-TUD/CTAB-GAN-Plus).
# Trains on the FULL training set (class-conditional); excluded on 4 datasets
# (fraud_detection, protein_homo, unsw_nb15: projected tens of hours; ecoli: 0%
# minority yield) -> runs on 12/16. Output: ../results/tables/raw_ctabgan.csv
# ENV: the CTAB-GAN+ repo cloned at CTABGAN_REPO + torch, scikit-learn, lightgbm.
# ---- Shared configuration and protocol (identical across run notebooks) ----
# This block is self-contained so each notebook runs independently in its own
# conda environment. All 16 datasets must sit in ../data as <name>.csv with a
# 'target' column (1 = minority/positive).
import os, time, warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (roc_auc_score, f1_score, precision_score, recall_score,
                             average_precision_score, balanced_accuracy_score, confusion_matrix)
warnings.filterwarnings("ignore")

DATA_DIR   = "../data"
TABLES_DIR = "../results/tables"
os.makedirs(TABLES_DIR, exist_ok=True)

# Evaluation protocol for every method: 5 repeated stratified 70/30 holdouts.
SEEDS = [42, 43, 44, 45, 46]
# Optional: set to a short list (e.g. ["ecoli", "pima_diabetes"]) to test quickly.
DATASETS_SUBSET = None

DATASETS = ["credit_default", "fraud_detection", "pima_diabetes", "ibm_attrition",
            "yeast_me2", "mammography", "abalone_19", "wine_quality", "ecoli",
            "pageblocks", "protein_homo", "satellite", "churn", "secom",
            "thyroid_sick", "unsw_nb15"]
CLASSIFIERS = ("RF", "LGBM", "MLP")
RAW_COLUMNS = ["dataset", "method", "classifier", "seed", "AUC", "PR_AUC", "F1",
               "Precision", "Recall", "G_mean", "Balanced_Acc",
               "generation_time", "train_time", "method_group"]

def load_dataset(name):
    """Read ../data/<name>.csv -> (X float64 ndarray, y int ndarray)."""
    df = pd.read_csv(os.path.join(DATA_DIR, f"{name}.csv"))
    bcols = df.select_dtypes(include="bool").columns
    if len(bcols):
        df[bcols] = df[bcols].astype("float64")
    X = df.drop(columns=["target"]).values.astype(float)
    y = df["target"].values.astype(int)
    return X, y

def get_classifier(name, variant, random_state):
    """variant: 'generative' (notebooks 03/04x config) or 'classic' (notebook 02)."""
    if name == "RF":
        return RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=3,
                                      class_weight="balanced", random_state=random_state, n_jobs=-1)
    if name == "LGBM":
        from lightgbm import LGBMClassifier
        if variant == "classic":
            return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                                  random_state=random_state, verbosity=-1, n_jobs=-1)
        return LGBMClassifier(n_estimators=100, learning_rate=0.05, num_leaves=31,
                              class_weight="balanced", random_state=random_state, n_jobs=-1, verbose=-1)
    if name == "MLP":
        if variant == "classic":
            return MLPClassifier(hidden_layer_sizes=(128, 64), activation="relu",
                                 alpha=0.001, random_state=random_state, max_iter=500)
        return MLPClassifier(hidden_layer_sizes=(128, 64), alpha=0.001,
                             max_iter=300, random_state=random_state)
    raise ValueError(name)

def g_mean_score(y_true, y_pred):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return float(np.sqrt(sens * spec))

def evaluate(model, X_test, y_test):
    yp = model.predict_proba(X_test)[:, 1]
    yd = model.predict(X_test)
    return {"AUC": round(roc_auc_score(y_test, yp), 4),
            "PR_AUC": round(average_precision_score(y_test, yp), 4),
            "F1": round(f1_score(y_test, yd, average="macro", zero_division=0), 4),
            "Precision": round(precision_score(y_test, yd, average="macro", zero_division=0), 4),
            "Recall": round(recall_score(y_test, yd, average="macro", zero_division=0), 4),
            "G_mean": round(g_mean_score(y_test, yd), 4),
            "Balanced_Acc": round(balanced_accuracy_score(y_test, yd), 4)}

def holdout_split_scale(X, y, seed):
    """Single stratified 70/30 holdout; StandardScaler fit on train only."""
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.30, stratify=y, random_state=seed)
    sc = StandardScaler()
    return sc.fit_transform(Xtr), sc.transform(Xte), ytr, yte

def append_rows(out_csv, rows):
    pd.DataFrame(rows, columns=RAW_COLUMNS).to_csv(
        out_csv, mode="a", header=not os.path.exists(out_csv), index=False)

def seed_everything(seed):
    np.random.seed(seed)
    try:
        import torch; torch.manual_seed(seed)
    except Exception:
        pass

def already_done(out_csv, method):
    """Return the set of (dataset, seed) already written for this method, so a
    re-run resumes instead of duplicating (append mode). Robust to a missing file."""
    done = set()
    if os.path.exists(out_csv):
        try:
            # keep_default_na=False so the literal method name "None" is NOT parsed as NaN
            prev = pd.read_csv(out_csv, usecols=["dataset", "method", "seed"], keep_default_na=False)
            prev = prev[prev["method"] == method]
            done = set((str(d), int(s)) for d, s in prev[["dataset", "seed"]].drop_duplicates().values)
        except Exception:
            pass
    return done

def run_oversampler(method, method_group, oversample_fn, out_csv, variant="generative",
                    skip_datasets=()):
    """Generic driver: for each seed x dataset, oversample then fit/evaluate 3 classifiers.
    oversample_fn(X_tr_scaled, y_tr, seed) -> (X_res, y_res, generation_time_or_None).
    - Resumes: skips (dataset, seed) already present in out_csv for this method.
    - Robust: a failure on one (dataset, seed) is logged and skipped, not fatal."""
    datasets = DATASETS_SUBSET or DATASETS
    done = already_done(out_csv, method)
    for seed in SEEDS:
        for ds in datasets:
            if ds in skip_datasets:
                continue
            if (ds, seed) in done:
                print(f"[skip-done] {method:12s} seed={seed} {ds}"); continue
            try:
                X, y = load_dataset(ds)
            except FileNotFoundError:
                print(f"[missing csv] {ds}"); continue
            try:
                Xtr, Xte, ytr, yte = holdout_split_scale(X, y, seed)
                seed_everything(seed)
                t0 = time.time()
                Xr, yr, gt = oversample_fn(Xtr, ytr, seed)
                gen_time = round(time.time() - t0, 3) if gt is None else round(gt, 3)
                rows = []
                for cn in CLASSIFIERS:
                    clf = get_classifier(cn, variant, random_state=seed)
                    t1 = time.time(); clf.fit(Xr, yr); tt = round(time.time() - t1, 3)
                    rows.append({"dataset": ds, "method": method, "classifier": cn, "seed": seed,
                                 **evaluate(clf, Xte, yte), "generation_time": gen_time,
                                 "train_time": tt, "method_group": method_group})
                append_rows(out_csv, rows)
                print(f"[ok] {method:12s} seed={seed} {ds:16s} "
                      f"AUC(RF/LGBM/MLP)={rows[0]['AUC']}/{rows[1]['AUC']}/{rows[2]['AUC']}")
            except Exception as e:
                print(f"[FAIL-skip] {method:12s} seed={seed} {ds:16s} -> {type(e).__name__}: {e}")
                continue

import sys, threading
CTABGAN_REPO = os.path.expanduser("~/CTAB-GAN-Plus")   # <-- set to your clone path
sys.path.insert(0, CTABGAN_REPO)
GENERATE_TIMEOUT_SEC = 600
OUT = os.path.join(TABLES_DIR, "raw_ctabgan.csv")
EXCLUDED = {"fraud_detection", "protein_homo", "unsw_nb15", "ecoli"}
_orig_cwd = os.getcwd()

class TimeoutException(Exception): pass

def run_with_timeout(func, timeout_sec=GENERATE_TIMEOUT_SEC):
    res, exc = {}, {}
    def tgt():
        try: res["v"] = func()
        except Exception as e: exc["e"] = e
    th = threading.Thread(target=tgt, daemon=True); th.start(); th.join(timeout_sec)
    if th.is_alive(): raise TimeoutException("timeout")
    if "e" in exc: raise exc["e"]
    return res.get("v")

def apply_ctabgan_plus(Xtr, ytr, seed, min_var=1e-8, max_retries=8):
    os.chdir(CTABGAN_REPO)
    try:
        from model.ctabgan import CTABGAN
        nmaj, nmin = int((ytr == 0).sum()), int((ytr == 1).sum()); need = nmaj - nmin
        if need <= 0: return Xtr, ytr, None
        nf = Xtr.shape[1]; cmask = Xtr.var(0) < min_var
        vidx, cidx = np.where(~cmask)[0], np.where(cmask)[0]
        cvals = Xtr[0, cidx] if len(cidx) else np.array([])
        cols = [f"f{i}" for i in range(len(vidx))]
        df = pd.DataFrame(Xtr[:, vidx], columns=cols); df["target"] = ytr
        os.makedirs("Real_Datasets", exist_ok=True)
        tmp = os.path.join("Real_Datasets", "ctabgan_temp_full.csv"); df.to_csv(tmp, index=False)
        syn = CTABGAN(raw_csv_path=tmp, test_ratio=0.20, categorical_columns=[], log_columns=[],
                      mixed_columns={}, general_columns=cols, non_categorical_columns=[],
                      integer_columns=[], problem_type={"Classification": "target"})
        syn.fit()
        parts, got, r = [], 0, 0
        while got < need and r < max_retries:
            sdf = run_with_timeout(syn.generate_samples)
            tr = np.round(sdf["target"].values).astype(int)
            Xraw = sdf[cols].values.astype(float); nanm = np.isnan(Xraw).any(1)
            keep = (tr == 1) & ~nanm
            if keep.sum() > 0: parts.append(Xraw[keep]); got += int(keep.sum())
            r += 1
        if got == 0: raise ValueError("no usable minority rows")
        Xv = np.vstack(parts); rng = np.random.RandomState(seed)
        idx = rng.choice(len(Xv), need, replace=len(Xv) < need); Xv = Xv[idx]
        Xs = np.zeros((need, nf)); Xs[:, vidx] = Xv
        if len(cidx): Xs[:, cidx] = cvals
        Xmaj, Xmin = Xtr[ytr == 0], Xtr[ytr == 1]
        return (np.vstack([Xmaj, Xmin, Xs]),
                np.concatenate([np.zeros(len(Xmaj)), np.ones(len(Xmin)), np.ones(len(Xs))]).astype(int), None)
    finally:
        os.chdir(_orig_cwd)

print("CTAB-GAN+ excluded datasets:", EXCLUDED)
run_oversampler("CTAB-GAN+", "new_generative",
                lambda Xtr, ytr, seed: apply_ctabgan_plus(Xtr, ytr, seed),
                OUT, skip_datasets=EXCLUDED)
print("DONE -> ", OUT)

CTAB-GAN+ excluded datasets: {'unsw_nb15', 'protein_homo', 'ecoli', 'fraud_detection'}
[skip-done] CTAB-GAN+    seed=42 credit_default
[skip-done] CTAB-GAN+    seed=42 pima_diabetes
[skip-done] CTAB-GAN+    seed=42 ibm_attrition
[skip-done] CTAB-GAN+    seed=42 yeast_me2
[skip-done] CTAB-GAN+    seed=42 mammography
[skip-done] CTAB-GAN+    seed=42 abalone_19
[skip-done] CTAB-GAN+    seed=42 wine_quality
[skip-done] CTAB-GAN+    seed=42 pageblocks
[skip-done] CTAB-GAN+    seed=42 satellite
[skip-done] CTAB-GAN+    seed=42 churn
[skip-done] CTAB-GAN+    seed=42 secom
[skip-done] CTAB-GAN+    seed=42 thyroid_sick
[skip-done] CTAB-GAN+    seed=43 credit_default
[skip-done] CTAB-GAN+    seed=43 pima_diabetes
[skip-done] CTAB-GAN+    seed=43 ibm_attrition
[skip-done] CTAB-GAN+    seed=43 yeast_me2
[skip-done] CTAB-GAN+    seed=43 mammography
[skip-done] CTAB-GAN+    seed=43 abalone_19
[skip-done] CTAB-GAN+    seed=43 wine_quality
[skip-done] CTAB-GAN+    seed=43 pageblocks
[skip-done] CTAB-G

100%|██████████| 150/150 [03:14<00:00,  1.30s/it]


Finished training in 197.36022019386292  seconds.
[FAIL-skip] CTAB-GAN+    seed=45 thyroid_sick     -> ValueError: no usable minority rows


100%|██████████| 150/150 [26:10<00:00, 10.47s/it]


Finished training in 1571.276368379593  seconds.
[ok] CTAB-GAN+    seed=46 credit_default   AUC(RF/LGBM/MLP)=0.7806/0.7833/0.6975


...

Finished training in 236.24199962615967  seconds.
[ok] CTAB-GAN+    seed=46 churn            AUC(RF/LGBM/MLP)=0.9309/0.9216/0.8988


100%|██████████| 150/150 [07:48<00:00,  3.12s/it]


Finished training in 469.1595513820648  seconds.
[ok] CTAB-GAN+    seed=46 secom            AUC(RF/LGBM/MLP)=0.7315/0.7755/0.7188


100%|██████████| 150/150 [03:18<00:00,  1.32s/it]


Finished training in 198.52496075630188  seconds.
[FAIL-skip] CTAB-GAN+    seed=46 thyroid_sick     -> ValueError: no usable minority rows
DONE ->  ../results/tables\raw_ctabgan.csv
